In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd

from sklearn import preprocessing
from sklearn import metrics
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import AgglomerativeClustering, KMeans

from factor_analyzer import FactorAnalyzer
from factor_analyzer.factor_analyzer import calculate_bartlett_sphericity

import matplotlib.pyplot as plt
import seaborn as sns

# **DATA**

In [ ]:
buildings = gpd.read_file('data/buildings_bk.shp')

In [ ]:
merged = pd.read_csv('data/merged.csv')
percentiles = pd.read_csv('data/percentiles.csv')

df = pd.concat([merged, percentiles], axis=1)
df = df.drop(columns=['eID', 'nID', 'bID', 'nodeID', 'mm_len', 'node_start', 'node_end'])
df = df.replace([np.inf, -np.inf], 0)
df = df.fillna(0)

scaler = preprocessing.StandardScaler()
cols = df.columns.tolist()[1:]
df[cols] = scaler.fit_transform(df[cols])

# **PCA DATA**

In [ ]:
df = pd.read_csv('data/merged.csv')

df = df.drop(columns=['eID', 'nID', 'bID', 'nodeID', 'mm_len', 'node_start', 'node_end'])
df = df.replace([np.inf, -np.inf], 0)
df = df.fillna(0)

scaler = preprocessing.StandardScaler()
cols = df.columns.tolist()[1:]
df[cols] = scaler.fit_transform(df[cols])

In [ ]:
pca = PCA()
principal_components = pca.fit_transform(df[cols])

principal_df = pd.DataFrame(data=principal_components)
principal_df.columns = ['PC' + str(i) for i in range(1, principal_df.shape[1]+1)]

principal_df = principal_df.iloc[:, :np.where(pca.explained_variance_ >= 1)[0][-1]]

principal_df_cols = principal_df.columns.tolist()

# **KMeans**

In [ ]:
def get_kmeans_scores(dataframe, variables_list):
    
    kmeans_clusters = pd.DataFrame()

    for i in range(2, 31):
        kmeans = KMeans(n_clusters=i)
        kmeans = kmeans.fit(dataframe[variables_list])

        kmeans_clusters['Clusters ' + str(i)] = kmeans.labels_
        
    
    silhouette_scores = []
    ch_scores = []
    db_scores = []

    for number_clusters in ["Clusters " + str(i) for i in range(2,31)]:
        silhouette_score = metrics.silhouette_score(dataframe[variables_list], kmeans_clusters[number_clusters])
        silhouette_scores.append((number_clusters, silhouette_score))

        ch_score = metrics.calinski_harabasz_score(dataframe[variables_list], kmeans_clusters[number_clusters])
        ch_scores.append((number_clusters, ch_score))
        
        db_score = metrics.davies_bouldin_score(dataframe[variables_list], kmeans_clusters[number_clusters])
        db_scores.append((number_clusters, db_score))


    silhouette_scores = pd.DataFrame(silhouette_scores, columns=['Number of Clusters', 'Silhouette Score']).set_index('Number of Clusters')
    ch_scores = pd.DataFrame(ch_scores, columns=['Number of Clusters', 'CH Score']).set_index('Number of Clusters')
    db_scores = pd.DataFrame(db_scores, columns=['Number of Clusters', 'DB Score']).set_index('Number of Clusters')
    
    scores = silhouette_scores.merge(ch_scores, on=['Number of Clusters']).merge(db_scores, on=['Number of Clusters'])

    return scores

In [ ]:
kmeans_scores = get_kmeans_scores(principal_df, principal_df_cols)

# **GMM**

In [ ]:
bic = pd.DataFrame(columns=['n', 'bic', 'run'])
ix = 0

n_components_range = range(2, 20) 
gmmruns = 1

for n_components in n_components_range:
    for i in range(gmmruns):
        gmm = GaussianMixture(n_components=n_components, covariance_type="full", max_iter=200, n_init=1, verbose=1)
        fitted = gmm.fit(df[cols])
        bicnum = gmm.bic(df[cols])
        bic.loc[ix] = [n_components, bicnum, i]
        ix += 1

In [ ]:
fig, ax = plt.subplots(figsize=(16, 16))
sns.lineplot(ax=ax, x='n', y='bic', data=bic)

In [ ]:
n = 5 
n_init = 5 

gmm = GaussianMixture(n_components=n, covariance_type="full", max_iter=200, n_init=n_init, verbose=1)
fitted = gmm.fit(df[cols])

In [ ]:
labels = gmm.predict(df[cols])
pd.Series(labels, index=df.index).to_csv('data/labels.csv', index=False)

In [ ]:
buildings.plot(labels, categorical=True, cmap='tab20', legend=True, figsize=(150, 150))
plt.savefig('building_cluster_map.jpg', dpi=300)